# Phase 7 — AutoML with PyCaret

**Purpose:** Benchmark all available classifiers via PyCaret's `compare_models`, tune the
best model, and compare against the Phase 6 proxy-risk baselines.

**Kernel:** SA Water (Python 3.11) — use `sa-water-py311` kernel.

**Input:** `data/clean/clean_master_sa2_v2.csv`

**Outputs:**
- `outputs/models/pycaret_best_model.pkl` — best PyCaret model pipeline
- `outputs/figures/07_leaderboard_f1.html` — all-model CV macro F1 leaderboard
- `outputs/figures/07_confusion_matrix.html` — test-set confusion matrix
- `outputs/figures/07_comparison_p6_vs_pycaret.html` — PyCaret vs Phase 6 side-by-side

**Phase 6 baselines to beat (SEIFA proxy-risk model, macro F1):**

| Model | CV Macro F1 | Test Macro F1 |
|---|---|---|
| LogReg | 0.616 ± 0.070 | 0.563 |
| RandomForest | 0.592 ± 0.102 | 0.618 |
| GradientBoosting | 0.615 ± 0.106 | 0.578 |

**Target:** `burden_tier_rel` (relative percentile tier within SA Water SA2s)

**PyCaret gotcha:** `setup()` resets the dataframe index — save `SA2_CODE21` before calling `setup()`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import pickle

import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, f1_score, classification_report

from pycaret.classification import (
    setup,
    compare_models,
    tune_model,
    pull,
    predict_model,
    save_model,
)

warnings.filterwarnings('ignore')

ROOT   = Path('..').resolve()
CLEAN  = ROOT / 'data' / 'clean'
FIGS   = ROOT / 'outputs' / 'figures'
MODELS = ROOT / 'outputs' / 'models'
MODELS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TIER_ORDER   = ['Low', 'Moderate', 'High', 'Critical']

print('Imports OK')

Imports OK


## 1. Load data

In [2]:
df_all = pd.read_csv(CLEAN / 'clean_master_sa2_v2.csv', dtype={'SA2_CODE21': str})
print('Loaded:', df_all.shape)
print('\nburden_tier_rel distribution:')
print(df_all['burden_tier_rel'].value_counts())

Loaded: (176, 37)

burden_tier_rel distribution:
burden_tier_rel
Moderate    84
Low         42
High        25
Critical    17
Unknown      8
Name: count, dtype: int64


## 2. Filter and prepare

Keep SA Water SA2s with valid burden ratios only.
Save `SA2_CODE21` / `SA2_NAME21` **before** `setup()` — PyCaret resets the dataframe index.

In [3]:
df = df_all[
    (df_all['provider_type'] == 'SA Water') &
    (df_all['burden_tier_rel'] != 'Unknown')
].copy().reset_index(drop=True)

print('After filtering to SA Water + known tiers:', df.shape)
print(df['burden_tier_rel'].value_counts())

sa2_meta = df[['SA2_CODE21', 'SA2_NAME21']].copy()

After filtering to SA Water + known tiers: (168, 37)
burden_tier_rel
Moderate    84
Low         42
High        25
Critical    17
Name: count, dtype: int64


## 3. Feature selection

Same SEIFA proxy-risk features as Phase 6 — no income, no bill (leakage exclusions).

In [4]:
FEATURE_COLS = [
    'irsd_score',
    'irsad_score',
    'ier_score',
    'ieo_score',
    'population',
    'AREASQKM21',
]
TARGET_COL = 'burden_tier_rel'

df_model = df[FEATURE_COLS + [TARGET_COL]].copy()
print('Model dataframe:', df_model.shape)
print('\nMissing values:')
print(df_model.isnull().sum())

Model dataframe: (168, 7)

Missing values:
irsd_score         3
irsad_score        3
ier_score          3
ieo_score          2
population         2
AREASQKM21         0
burden_tier_rel    0
dtype: int64


## 4. PyCaret setup

- 5-fold stratified CV
- Median imputation for SA2s with missing SEIFA scores
- SMOTE imbalance correction (17 Critical vs 84 Moderate)
- 80/20 train/test split

In [5]:
clf_setup = setup(
    data=df_model,
    target=TARGET_COL,
    session_id=RANDOM_STATE,
    fold=5,
    fold_strategy='stratifiedkfold',
    numeric_imputation='median',
    fix_imbalance=True,
    fix_imbalance_method='smote',
    train_size=0.8,
    verbose=False,
)
print('PyCaret setup complete.')
print(f'Training rows : {clf_setup.get_config("X_train").shape[0]}')
print(f'Test rows     : {clf_setup.get_config("X_test").shape[0]}')
print(f'Features      : {clf_setup.get_config("X_train").shape[1]}')

PyCaret setup complete.
Training rows : 134
Test rows     : 34
Features      : 6


## 5. Compare all models — sorted by macro F1

In [6]:
top3 = compare_models(
    sort='F1',
    n_select=3,
    verbose=True,
)
leaderboard = pull()
print('\nLeaderboard (sorted by CV Macro F1):')
print(
    leaderboard[['Model', 'Accuracy', 'AUC', 'Recall', 'Prec.', 'F1', 'Kappa', 'MCC']]
    .to_string(index=False)
)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.7011,0.8686,0.7011,0.7231,0.6960,0.5582,0.5673,0.0780
gbc,Gradient Boosting Classifier,0.6937,0.0000,0.6937,0.6916,0.6853,0.5323,0.5366,0.1800
rf,Random Forest Classifier,0.6860,0.8846,0.6860,0.6883,0.6779,0.5233,0.5287,0.1020
qda,Quadratic Discriminant Analysis,0.6869,0.0000,0.6869,0.6897,0.6739,0.5141,0.5192,0.0280
lightgbm,Light Gradient Boosting Machine,0.6792,0.8598,0.6792,0.6883,0.6706,0.5079,0.5142,3.0980
nb,Naive Bayes,0.6567,0.8570,0.6567,0.6681,0.6464,0.4949,0.5083,1.2140
lda,Linear Discriminant Analysis,0.6040,0.0000,0.6040,0.6578,0.6156,0.4292,0.4412,0.0220
dt,Decision Tree Classifier,0.5969,0.7032,0.5969,0.6163,0.5897,0.4037,0.4119,1.4440
ridge,Ridge Classifier,0.5749,0.0000,0.5749,0.6508,0.5744,0.4131,0.4353,0.0380
lr,Logistic Regression,0.5160,0.0000,0.5160,0.6036,0.5210,0.3219,0.3386,1.4120



Leaderboard (sorted by CV Macro F1):
                          Model  Accuracy    AUC  Recall  Prec.     F1  Kappa    MCC
         Extra Trees Classifier    0.7011 0.8686  0.7011 0.7231 0.6960 0.5582 0.5673
   Gradient Boosting Classifier    0.6937 0.0000  0.6937 0.6916 0.6853 0.5323 0.5366
       Random Forest Classifier    0.6860 0.8846  0.6860 0.6883 0.6779 0.5233 0.5287
Quadratic Discriminant Analysis    0.6869 0.0000  0.6869 0.6897 0.6739 0.5141 0.5192
Light Gradient Boosting Machine    0.6792 0.8598  0.6792 0.6883 0.6706 0.5079 0.5142
                    Naive Bayes    0.6567 0.8570  0.6567 0.6681 0.6464 0.4949 0.5083
   Linear Discriminant Analysis    0.6040 0.0000  0.6040 0.6578 0.6156 0.4292 0.4412
       Decision Tree Classifier    0.5969 0.7032  0.5969 0.6163 0.5897 0.4037 0.4119
               Ridge Classifier    0.5749 0.0000  0.5749 0.6508 0.5744 0.4131 0.4353
            Logistic Regression    0.5160 0.0000  0.5160 0.6036 0.5210 0.3219 0.3386
           Ada Boost Classi

In [7]:
lb_plot = leaderboard[['Model', 'F1']].copy().sort_values('F1', ascending=True)

# Phase 6 best CV F1 for reference line
P6_BEST_CV_F1 = 0.616

fig = go.Figure(go.Bar(
    x=lb_plot['F1'],
    y=lb_plot['Model'],
    orientation='h',
    marker=dict(
        color=lb_plot['F1'],
        colorscale='Teal',
        cmin=0.3,
        cmax=1.0,
        showscale=True,
    ),
    text=lb_plot['F1'].round(3),
    textposition='outside',
))
fig.add_vline(
    x=P6_BEST_CV_F1,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Phase 6 best CV ({P6_BEST_CV_F1})',
    annotation_position='top right',
)
fig.update_layout(
    title='PyCaret AutoML — 5-Fold CV Macro F1 (SEIFA proxy-risk features)',
    xaxis_title='Mean CV Macro F1',
    xaxis=dict(range=[0, 1.15]),
    template='plotly_white',
    width=900,
    height=600,
)
fig.write_html(str(FIGS / '07_leaderboard_f1.html'))
print('Saved 07_leaderboard_f1.html')

Saved 07_leaderboard_f1.html


## 6. Tune best model

In [8]:
best_model = top3[0] if isinstance(top3, list) else top3
print(f'Tuning: {type(best_model).__name__}')

tuned_model = tune_model(
    best_model,
    optimize='F1',
    n_iter=50,
    verbose=False,
)
tuned_cv = pull()
print('\nTuned model CV scores:')
print(
    tuned_cv[['Accuracy', 'AUC', 'Recall', 'Prec.', 'F1', 'Kappa', 'MCC']]
    .to_string()
)

Tuning: ExtraTreesClassifier



Tuned model CV scores:
      Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                          
0       0.7778  0.8897  0.7778  0.7608  0.7637  0.6408  0.6460
1       0.6296  0.8650  0.6296  0.6843  0.6295  0.4664  0.4826
2       0.6667  0.8907  0.6667  0.6829  0.6603  0.5226  0.5332
3       0.6667  0.8837  0.6667  0.7094  0.6810  0.5169  0.5222
4       0.6923  0.8662  0.6923  0.7070  0.6904  0.5294  0.5381
Mean    0.6866  0.8791  0.6866  0.7089  0.6850  0.5352  0.5444
Std     0.0498  0.0112  0.0498  0.0282  0.0446  0.0573  0.0544


## 7. Evaluate on hold-out test set

In [9]:
test_preds = predict_model(tuned_model)
print('Test predictions shape:', test_preds.shape)

y_true = test_preds[TARGET_COL]
y_pred = test_preds['prediction_label']

test_macro_f1 = f1_score(y_true, y_pred, average='macro', labels=TIER_ORDER, zero_division=0)
print(f'\nTest Macro F1 : {test_macro_f1:.3f}')
print('\nClassification Report:')
print(classification_report(y_true, y_pred, labels=TIER_ORDER, zero_division=0))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.6176,0.8733,0.6176,0.6303,0.6206,0.4091,0.4108


Test predictions shape: (34, 9)

Test Macro F1 : 0.585

Classification Report:
              precision    recall  f1-score   support

         Low       0.71      0.56      0.62         9
    Moderate       0.67      0.71      0.69        17
        High       0.33      0.40      0.36         5
    Critical       0.67      0.67      0.67         3

    accuracy                           0.62        34
   macro avg       0.60      0.58      0.59        34
weighted avg       0.63      0.62      0.62        34



In [10]:
cm = confusion_matrix(y_true, y_pred, labels=TIER_ORDER)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig = go.Figure(data=go.Heatmap(
    z=cm_norm,
    x=[f'Pred: {t}' for t in TIER_ORDER],
    y=[f'True: {t}' for t in TIER_ORDER],
    colorscale='Blues',
    zmin=0, zmax=1,
    text=[[f'{cm[i][j]}<br>({cm_norm[i][j]:.0%})' for j in range(4)] for i in range(4)],
    texttemplate='%{text}',
))
fig.update_layout(
    title=f'Confusion Matrix — {type(tuned_model).__name__} (test set)',
    xaxis_title='Predicted', yaxis_title='Actual',
    template='plotly_white', width=600, height=500,
)
fig.write_html(str(FIGS / '07_confusion_matrix.html'))
print('Saved 07_confusion_matrix.html')

Saved 07_confusion_matrix.html


## 8. Compare PyCaret best vs Phase 6 baselines

In [11]:
try:
    tuned_cv_f1 = float(tuned_cv.loc['Mean', 'F1'])
except (KeyError, TypeError):
    numeric_mask = tuned_cv['F1'].apply(lambda x: isinstance(x, (int, float)))
    tuned_cv_f1 = float(tuned_cv.loc[numeric_mask, 'F1'].mean())

model_name = type(tuned_model).__name__
print(f'PyCaret best model : {model_name}')
print(f'Tuned CV Macro F1  : {tuned_cv_f1:.3f}')
print(f'Test Macro F1      : {test_macro_f1:.3f}')

comparison = pd.DataFrame({
    'Model': [
        'LogReg (Phase 6)',
        'RandomForest (Phase 6)',
        'GradientBoosting (Phase 6)',
        f'PyCaret {model_name} (Phase 7)',
    ],
    'CV Macro F1':   [0.616, 0.592, 0.615, round(tuned_cv_f1, 3)],
    'Test Macro F1': [0.563, 0.618, 0.578, round(test_macro_f1, 3)],
})
print('\n' + comparison.to_string(index=False))

PyCaret best model : ExtraTreesClassifier
Tuned CV Macro F1  : 0.685
Test Macro F1      : 0.585

                                 Model  CV Macro F1  Test Macro F1
                      LogReg (Phase 6)        0.616          0.563
                RandomForest (Phase 6)        0.592          0.618
            GradientBoosting (Phase 6)        0.615          0.578
PyCaret ExtraTreesClassifier (Phase 7)        0.685          0.585


In [12]:
fig = go.Figure()
fig.add_trace(go.Bar(
    name='CV Macro F1', x=comparison['Model'], y=comparison['CV Macro F1'],
    marker_color='#636EFA', text=comparison['CV Macro F1'].round(3), textposition='outside',
))
fig.add_trace(go.Bar(
    name='Test Macro F1', x=comparison['Model'], y=comparison['Test Macro F1'],
    marker_color='#EF553B', text=comparison['Test Macro F1'].round(3), textposition='outside',
))
fig.update_layout(
    barmode='group',
    title='Phase 6 Baseline vs PyCaret AutoML — Macro F1 (SEIFA proxy-risk features)',
    yaxis_title='Macro F1',
    yaxis=dict(range=[0, 1.2]),
    template='plotly_white', width=900, height=500,
)
fig.write_html(str(FIGS / '07_comparison_p6_vs_pycaret.html'))
print('Saved 07_comparison_p6_vs_pycaret.html')

Saved 07_comparison_p6_vs_pycaret.html


## 9. Save best model

In [13]:
pycaret_model_path = str(MODELS / 'pycaret_best_model')
save_model(tuned_model, pycaret_model_path)
print(f'Saved: {pycaret_model_path}.pkl')

print('\nAll model outputs:')
for p in sorted(MODELS.glob('*.pkl')):
    print(f'  {p.name}')

Transformation Pipeline and Model Successfully Saved


Saved: C:\Users\mussa\OneDrive\Desktop\Projects\sa_water_project\outputs\models\pycaret_best_model.pkl

All model outputs:
  baseline_gb.pkl
  baseline_logreg.pkl
  baseline_rf.pkl
  label_encoder.pkl
  pycaret_best_model.pkl


## 10. Phase summary

**What PyCaret adds:** Systematic benchmark of all sklearn classifiers, automated
hyperparameter tuning, SMOTE per-fold.

**Expected result with SEIFA-only features:** Best AutoML model likely scores in the
0.6–0.75 macro F1 range — realistic for a proxy-risk problem where SEIFA correlates
with income but doesn't perfectly determine it.

**Next:** Phase 8 — Simulation engine.